### Imports and Connection

In [8]:
import duckdb
import pandas as pdb

con = duckdb.connect("../data/processed/recipeiq.duckdb", read_only=True)
print("Connected to RecipeIQ warehouse ✓")

recipe_count = con.execute("SELECT COUNT(*) FROM recipes").fetchone()[0]
review_count = con.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]

print(f"Recipes: {recipe_count:,}")
print(f"Reviews: {review_count:,}")

Connected to RecipeIQ warehouse ✓
Recipes: 522,517
Reviews: 1,401,768


### Top 10 Recipe Categories by Count

In [9]:
df = con.execute("""
    SELECT
        RecipeCategory,
        COUNT(*) AS recipe_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
    FROM recipes
    WHERE RecipeCategory IS NOT NULL
    GROUP BY RecipeCategory
    ORDER BY recipe_count DESC
    LIMIT 10
""").fetchdf()

print("Top 10 Recipe Categories")
print("=" * 50)
print(df.to_string(index=False))

Top 10 Recipe Categories
RecipeCategory  recipe_count  percentage
       Dessert         62072        11.9
  Lunch/Snacks         32586         6.2
 One Dish Meal         31345         6.0
     Vegetable         27231         5.2
     Breakfast         21101         4.0
     Beverages         16076         3.1
       Chicken         13249         2.5
          Meat         13131         2.5
        Breads         12804         2.5
          Pork         12603         2.4


### Average Rating By Complexity

In [10]:
df = con.execute("""
    SELECT
        CASE complexity_score
            WHEN 1 THEN '🟢 Simple'
            WHEN 2 THEN '🟡 Medium'
            WHEN 3 THEN '🔴 Complex'
        END AS difficulty,
        COUNT(*) AS recipe_count,
        ROUND(AVG(AggregatedRating), 2) AS avg_rating,
        ROUND(AVG(ReviewCount), 1) AS avg_reviews,
        ROUND(AVG(calories_per_serving), 0) AS avg_cal_per_serving
    FROM recipes
    WHERE AggregatedRating IS NOT NULL
    GROUP BY complexity_score
    ORDER BY complexity_score
""").fetchdf()

print("Rating by Complexity")
print("=" * 50)
print(df.to_string(index=False))

Rating by Complexity
difficulty  recipe_count  avg_rating  avg_reviews  avg_cal_per_serving
  🟢 Simple         35779        4.66          4.3                107.0
  🟡 Medium        171324        4.62          5.4                 93.0
 🔴 Complex         62191        4.64          5.8                101.0


### Nutrition Profile — Healthiest vs Unhealthiest Categories

In [11]:
df = con.execute ("""
    SELECT
        RecipeCategory,
        COUNT(*) AS num,
        ROUND(AVG(calories_per_serving), 0) AS avg_cal,
        ROUND(AVG(protein_per_serving), 1) AS avg_protein,
        ROUND(AVG(fat_per_serving), 1) AS avg_fat,
        ROUND(AVG(sugar_per_serving), 1) AS avg_sugar
    FROM recipes
    WHERE RecipeCategory IS NOT NULL
    AND calories_per_serving IS NOT NULL
    AND calories_per_serving < 2000 -- Exclude likely data errors
    GROUP BY RecipeCategory
    HAVING COUNT(*) >= 100 -- Only categories with enough data
    ORDER BY avg_cal ASC
    LIMIT 15

""").fetchdf()

print("Healthiest Recipe Categories (by calories)")
print("=" * 60)
print(df.to_string(index=False))

Healthiest Recipe Categories (by calories)
  RecipeCategory  num  avg_cal  avg_protein  avg_fat  avg_sugar
For Large Groups  531     11.0          0.4      0.6        0.4
      Bar Cookie 3816     19.0          0.3      1.0        1.6
    Drop Cookies 2820     24.0          0.4      1.2        1.9
    Quick Breads 5713     29.0          0.6      1.2        1.7
           Candy 2370     36.0          0.4      1.5        4.4
         Gelatin  922     36.0          0.8      1.5        4.2
          Scones  709     40.0          0.9      1.8        1.4
         Berries  249     40.0          0.9      1.5        4.4
  Punch Beverage 1315     42.0          0.3      0.4        5.2
    Yeast Breads 2531     42.0          1.1      1.1        1.0
          Melons  214     44.0          1.2      1.5        5.6
         Spreads 2164     45.0          1.6      3.2        0.8
    Thanksgiving  138     45.0          1.5      2.3        1.4
      Cheesecake 2637     47.0          0.9      3.1        2

### Rating Distribution — Are Most Recipes Highly Rated?

In [12]:
df = con.execute("""
    SELECT
        ROUND(AggregatedRating, 0) AS rating_bucket,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
    FROM recipes
    WHERE AggregatedRating IS NOT NULL
    GROUP BY rating_bucket
    ORDER BY rating_bucket
""").fetchdf()

print("Rating Distribution")
print("=" * 50)
print(df.to_string(index=False))

Rating Distribution
 rating_bucket  count  percentage
           1.0   1677         0.6
           2.0   2125         0.8
           3.0   9839         3.7
           4.0  46807        17.4
           5.0 208846        77.6


### Most Reviewed Recipes — The "Greatest Hits

In [15]:
df = con.execute("""
    SELECT
        Name,
        RecipeCategory,
        CAST(ReviewCount AS INTEGER) AS reviews,
        AggregatedRating AS rating,
        ingredient_count,
        ROUND(calories_per_serving, 0) AS cal_per_serving,
        CASE complexity_score
            WHEN 1 THEN 'Simple'
            WHEN 2 THEN 'Medium'
            WHEN 3 THEN 'Complex'
        END AS difficulty
    FROM recipes
    WHERE ReviewCount IS NOT NULL
    ORDER BY ReviewCount DESC
    LIMIT 20
""").fetchdf()

print("Top 20 Most Reviewed Recipes")
print("=" * 70)
print(df.to_string(index=False))

Top 20 Most Reviewed Recipes
                                                      Name      RecipeCategory  reviews  rating  ingredient_count  cal_per_serving difficulty
                                           Bourbon Chicken      Chicken Breast     3063     5.0                10            130.0     Medium
                                         Best Banana Bread        Quick Breads     2273     5.0                 8             27.0     Medium
                                To Die for Crock Pot Roast       One Dish Meal     1692     5.0                 2             37.0     Medium
     Crock-Pot Chicken With Black Beans &amp; Cream Cheese       One Dish Meal     1657     4.5                 5            170.0     Medium
                                Creamy Cajun Chicken Pasta      Chicken Breast     1586     5.0                12            360.0     Medium
                                    Oatmeal Raisin Cookies        Drop Cookies     1410     5.0                11      

In [16]:
con.close()
print("Connection closed ✓")

Connection closed ✓
